# MoMo-FDVS logical PR14 — frozen transaction features

This owner-operated notebook builds one registered structured source at a time into content-hashed private Parquet shards. It creates chronological partitions and causal features but performs no model fitting, calibration, threshold selection, locked-test analysis or promotion.

In [ ]:
RUN_PROFILE = "smoke"
TARGET_COMMIT = "REPLACE_WITH_PUSHED_PR14_SHA"
REPOSITORY_URL = "https://github.com/davidagyekum/momo-fraud-detection.git"
DRIVE_ROOT = "/content/drive/MyDrive/momo-fraud"
VM_ROOT = "/content/momo-work"
NOTEBOOK_PATH = "ml/notebooks/colab/03_build_transaction_features.ipynb"
DATASET_ID = "paysim"  # run paysim, momtsim-v1 and momtsim-v2 separately
assert RUN_PROFILE == "smoke"
assert DATASET_ID in {"paysim", "momtsim-v1", "momtsim-v2"}
assert len(TARGET_COMMIT) == 40 and TARGET_COMMIT != "REPLACE_WITH_PUSHED_PR14_SHA"

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
repo = Path(VM_ROOT) / "repo"
repo.parent.mkdir(parents=True, exist_ok=True)
if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "fetch", "--prune", "origin"], check=True)
else:
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", TARGET_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", str(repo / "ml/requirements-runtime.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--editable", str(repo / "ml")], check=True)
sys.path.insert(0, str(repo / "ml/src"))

In [ ]:
import json
from momo_fdvs_ml.colab import ColabPaths, colab_preflight_report, repository_state
from momo_fdvs_ml.execution import ExecutionProfile

paths = ColabPaths(drive_root=Path(DRIVE_ROOT), vm_root=Path(VM_ROOT))
preflight = colab_preflight_report(repo, paths=paths, profile=ExecutionProfile.SMOKE, notebook=NOTEBOOK_PATH, require_colab=True)
assert preflight["git"]["commit"] == TARGET_COMMIT
assert preflight["git"]["dirty"] is False
assert preflight["full_training_executed"] is False
print(json.dumps({"commit": TARGET_COMMIT, "profile": RUN_PROFILE, "dataset_id": DATASET_ID, "full_training_executed": False}, indent=2, sort_keys=True))

In [ ]:
import os
import shutil
from momo_fdvs_ml.transaction_etl import TransactionBuildSpec, build_transaction_parquet_dataset

source_root = Path(DRIVE_ROOT) / "datasets"
configs = {
    "paysim": {"filename": "paysim-ealaxi-v2-f7eef9ffad5c.zip", "sha256": "f7eef9ffad5cfa64a034143a5c9b30491d189420b273d5ad5723ca40b596613d", "rows": 6362620, "positives": 8213, "entrypoint": "PS_20174392719_1491204439457_log.csv"},
    "momtsim-v1": {"filename": "synthetic_mobile_money_transaction_dataset.csv", "sha256": "da951eb95735da96271740a3e66b676b342d3831ce3111cd19dbfa020d3bd0a7", "rows": 1720181, "positives": 175518, "entrypoint": None},
    "momtsim-v2": {"filename": "momtsim-v2-derived-exact-dedup-v1.csv", "sha256": "642fcb2ba7c9cbfffb933729d118f426fefddcbaabbf002793807be169fe80cd", "rows": 4225938, "positives": 2233118, "entrypoint": None},
}
config = configs[DATASET_ID]
source_path = source_root / config["filename"]
assert source_path.is_file(), "Upload the exact registered source to the configured private Drive path."
vm_output = Path(VM_ROOT) / "outputs/pr14" / f"{DATASET_ID}-{config['sha256'][:12]}"
drive_output = Path(DRIVE_ROOT) / "runs/pr14-transaction-features" / f"{DATASET_ID}-{config['sha256'][:12]}"
if drive_output.exists():
    report = json.loads((drive_output / "build-report.json").read_text(encoding="utf-8"))
    assert report["source_sha256"] == config["sha256"]
else:
    report = build_transaction_parquet_dataset(source_path=source_path, output_path=vm_output, spec=TransactionBuildSpec(dataset_id=DATASET_ID, source_sha256=config["sha256"], expected_row_count=config["rows"], expected_positive_count=config["positives"], minimum_partition_positives=100, shard_size=100000, entrypoint=config["entrypoint"]))
    drive_output.parent.mkdir(parents=True, exist_ok=True)
    drive_staging = drive_output.parent / f".{drive_output.name}.tmp"
    if drive_staging.exists():
        raise RuntimeError("A prior incomplete Drive staging directory needs manual review.")
    shutil.copytree(vm_output, drive_staging)
    os.replace(drive_staging, drive_output)
safe_summary = {key: report[key] for key in ("dataset_id", "source_sha256", "split_manifest_sha256", "preprocessor_sha256", "row_count", "positive_count", "partitions", "elapsed_seconds", "peak_memory_bytes", "locked_test_sealed", "locked_test_accessed_for_decisions", "training_executed")}
print(json.dumps(safe_summary, indent=2, sort_keys=True))
assert report["locked_test_sealed"] is True
assert report["locked_test_accessed_for_decisions"] is False
assert report["training_executed"] is False

## Stop boundary

Run this notebook separately for each registered structured source and report only the safe summary. Stop after all three frozen split/feature bundles are hash-verified. Do not load locked-test labels, fit or compare models, calibrate scores, choose thresholds or promote artifacts. Logical PR15 transaction training starts only after Codex reviews these PR14 manifests and explicitly hands the owner the pinned Colab training notebook.